# EEGNet — trening w Google Colab

Przed startem wybierz **Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU**. Uruchamiaj komórki po kolei.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Konfiguracja
GITHUB_REPO = 'https://github.com/taf4you2/biai.git'
GITHUB_BRANCH = 'codex/eeg-pipeline-qc'
DRIVE_ZIP = '/content/drive/MyDrive/biai/data/biai_eeg_qc_0_0p8.zip'
DRIVE_RESULTS = '/content/drive/MyDrive/biai/results/eegnet_mole_colab'

TEST_PARTICIPANT = 'mole'
SMOKE_EPOCHS = 1
FULL_EPOCHS = 30
BATCH_SIZE = 256

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile

repo_dir = Path('/content/biai')
local_zip = Path('/content/biai_eeg_qc_0_0p8.zip')

if repo_dir.exists():
    shutil.rmtree(repo_dir)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPO, str(repo_dir)], check=True)

drive_zip = Path(DRIVE_ZIP)
if not drive_zip.is_file():
    raise FileNotFoundError(f'Nie znaleziono paczki na Drive: {drive_zip}')
shutil.copy2(drive_zip, local_zip)

with zipfile.ZipFile(local_zip) as archive:
    archive.extractall(repo_dir)

dataset_dir = repo_dir / 'event_epoch_multisession_image_on_0_0p8_qc'
print('Repo:', repo_dir)
print('Dataset:', dataset_dir)

In [ ]:
subprocess.run([
    'python', str(repo_dir / 'scripts/verify_colab_dataset.py'),
    '--dataset-dir', str(dataset_dir),
], check=True)

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU nie jest aktywne. Zmień typ środowiska wykonawczego na GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Test jednej epoki

Ta komórka sprawdza cały pipeline przed dłuższym treningiem.

In [ ]:
smoke_output = Path('/content/eegnet_smoke')
if smoke_output.exists():
    shutil.rmtree(smoke_output)

smoke_command = [
    'python', str(repo_dir / 'scripts/train_eegnet.py'),
    '--dataset-dir', str(dataset_dir),
    '--output-dir', str(smoke_output),
    '--split', 'participant_image',
    '--test-participant', TEST_PARTICIPANT,
    '--val-split', 'participant',
    '--normalization', 'participant',
    '--balanced-sampler', 'category_participant',
    '--only-qc-accepted',
    '--epochs', str(SMOKE_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
]
subprocess.run(smoke_command, cwd=repo_dir, check=True)

## Pełny trening

Uruchom po pomyślnym teście. Wyniki zostaną skopiowane na Google Drive.

In [ ]:
full_output = Path('/content/eegnet_mole_colab')
if full_output.exists():
    shutil.rmtree(full_output)

full_command = [
    'python', str(repo_dir / 'scripts/train_eegnet.py'),
    '--dataset-dir', str(dataset_dir),
    '--output-dir', str(full_output),
    '--split', 'participant_image',
    '--test-participant', TEST_PARTICIPANT,
    '--val-split', 'participant',
    '--normalization', 'participant',
    '--balanced-sampler', 'category_participant',
    '--only-qc-accepted',
    '--epochs', str(FULL_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
]
subprocess.run(full_command, cwd=repo_dir, check=True)

drive_results = Path(DRIVE_RESULTS)
drive_results.parent.mkdir(parents=True, exist_ok=True)
if drive_results.exists():
    shutil.rmtree(drive_results)
shutil.copytree(full_output, drive_results)
print('Wyniki zapisane w:', drive_results)